# Envoirnment Setup and Data Loading


In [ ]:
# Install required libraries
!pip -q install transformers datasets accelerate evaluate scikit-learn sentencepiece

import torch
import transformers
import sklearn
import pandas as pd
import numpy as np

print("="*50)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Transformers:", transformers.__version__)
print("="*50)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("complaints.csv")

print(df.head())

print(df.shape)

print(df.columns)

print(df["Department"].value_counts())

# Data Preprocessing

In [ ]:
import json
from sklearn.preprocessing import LabelEncoder

# Remove ID column if it exists
if "ID" in df.columns:
    df = df.drop(columns=["ID"])

# Remove missing values
df = df.dropna()

# Remove duplicate rows
df = df.drop_duplicates()

# Strip whitespace
df["Complaint"] = df["Complaint"].astype(str).str.strip()
df["Department"] = df["Department"].astype(str).str.strip()

# Encode department labels
label_encoder = LabelEncoder()

df["labels"] = label_encoder.fit_transform(df["Department"])

# Save label mapping
label_mapping = {
    int(v): k
    for v, k in enumerate(label_encoder.classes_)
}

with open("label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=4)

print("Dataset cleaned successfully")
print("Total Records :", len(df))
print("\nDepartments")

for i, name in label_mapping.items():
    print(i, "->", name)

# Train, Validation, Test Spill

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["labels"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["labels"],
    random_state=42
)

print("Training :", len(train_df))
print("Validation :", len(valid_df))
print("Testing :", len(test_df))

# Load DistilBERT Tokenizer

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer Loaded Successfully")

# Create Custom Dataset

In [ ]:
import torch
from torch.utils.data import Dataset

class ComplaintDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length=128):

        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        complaint = self.df.loc[index, "Complaint"]
        label = int(self.df.loc[index, "labels"])

        encoding = self.tokenizer(
            complaint,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

        return item

# Create Dataset Objects

In [ ]:
train_dataset = ComplaintDataset(train_df, tokenizer)
valid_dataset = ComplaintDataset(valid_df, tokenizer)
test_dataset = ComplaintDataset(test_df, tokenizer)

print("Train Dataset :", len(train_dataset))
print("Validation Dataset :", len(valid_dataset))
print("Test Dataset :", len(test_dataset))

# Create DataLoaders

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print("DataLoaders Created Successfully")

# Verify Everything

In [ ]:
batch = next(iter(train_loader))

print(batch.keys())

print()

print(batch["input_ids"].shape)

print(batch["attention_mask"].shape)

print(batch["labels"].shape)

# Load DistilBERT Model

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = len(label_encoder.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(device)

In [ ]:
import torch

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [ ]:
from transformers import get_linear_schedule_with_warmup

epochs = 3

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print("Total Training Steps:", total_steps)

# Import Libraries

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
def train_one_epoch(model, dataloader, optimizer, scheduler, device):

    model.train()

    total_loss = 0

    predictions = []
    actual_labels = []

    progress_bar = tqdm(dataloader)

    for batch in progress_bar:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        logits = outputs.logits

        loss.backward()

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())

        actual_labels.extend(labels.cpu().numpy())

        progress_bar.set_description(f"Loss : {loss.item():.4f}")

    accuracy = accuracy_score(actual_labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        actual_labels,
        predictions,
        average="weighted"
    )

    return (
        total_loss / len(dataloader),
        accuracy,
        precision,
        recall,
        f1
    )

In [ ]:
def evaluate(model, dataloader, device):

    model.eval()

    total_loss = 0

    predictions = []
    actual_labels = []

    with torch.no_grad():

        for batch in tqdm(dataloader):

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())

            actual_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(actual_labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        actual_labels,
        predictions,
        average="weighted"
    )

    return (
        total_loss / len(dataloader),
        accuracy,
        precision,
        recall,
        f1
    )

In [ ]:
best_accuracy = 0

epochs = 3

for epoch in range(epochs):

    print("="*60)
    print(f"Epoch {epoch+1}/{epochs}")
    print("="*60)

    train_loss, train_acc, train_pre, train_rec, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device
    )

    val_loss, val_acc, val_pre, val_rec, val_f1 = evaluate(
        model,
        valid_loader,
        device
    )

    print(f"\nTrain Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_acc:.4f}")

    print(f"\nValidation Loss : {val_loss:.4f}")
    print(f"Validation Acc  : {val_acc:.4f}")

    print(f"Precision       : {val_pre:.4f}")
    print(f"Recall          : {val_rec:.4f}")
    print(f"F1 Score        : {val_f1:.4f}")

    if val_acc > best_accuracy:

        best_accuracy = val_acc

        model.save_pretrained("complaint_classifier")

        tokenizer.save_pretrained("complaint_classifier")

        print("\n✅ Best Model Saved")

# Start Training

In [ ]:
best_accuracy = 0

epochs = 3

for epoch in range(epochs):

    print("="*60)
    print(f"Epoch {epoch+1}/{epochs}")
    print("="*60)

    train_loss, train_acc, train_pre, train_rec, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device
    )

    val_loss, val_acc, val_pre, val_rec, val_f1 = evaluate(
        model,
        valid_loader,
        device
    )

    print(f"\nTrain Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_acc:.4f}")

    print(f"\nValidation Loss : {val_loss:.4f}")
    print(f"Validation Acc  : {val_acc:.4f}")

    print(f"Precision       : {val_pre:.4f}")
    print(f"Recall          : {val_rec:.4f}")
    print(f"F1 Score        : {val_f1:.4f}")

    if val_acc > best_accuracy:

        best_accuracy = val_acc

        model.save_pretrained("complaint_classifier")

        tokenizer.save_pretrained("complaint_classifier")

        print("\n✅ Best Model Saved")

# Evaluate on the Test Dataset

In [ ]:
test_loss, test_acc, test_pre, test_rec, test_f1 = evaluate(
    model,
    test_loader,
    device
)

print("=" * 50)
print("TEST RESULTS")
print("=" * 50)

print(f"Test Loss      : {test_loss:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Precision      : {test_pre:.4f}")
print(f"Recall         : {test_rec:.4f}")
print(f"F1 Score       : {test_f1:.4f}")

# Test the Model with Real Complaints

In [ ]:
import torch
import json

# Load label mapping
with open("label_mapping.json", "r") as f:
    label_mapping = json.load(f)

# Reverse mapping
id_to_label = {int(k): v for k, v in label_mapping.items()}


def predict_department(complaint):

    model.eval()

    encoding = tokenizer(
        complaint,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    probabilities = torch.softmax(outputs.logits, dim=1)

    confidence, prediction = torch.max(probabilities, dim=1)

    return {
        "department": id_to_label[prediction.item()],
        "confidence": round(confidence.item() * 100, 2)
    }

In [ ]:
complaint = "There is no drinking water in Anna Nagar for the last three days."

result = predict_department(complaint)

print(result)

In [ ]:
predict_department("Street lights are not working in my area.")
predict_department("Garbage has not been collected for five days.")
predict_department("The road is full of potholes.")
predict_department("Someone demanded a bribe for approving my application.")

In [ ]:
test_complaints = [
    "There is no drinking water in my street for the past 4 days.",
    "The street lights are not working near the bus stand.",
    "Garbage has not been collected for one week.",
    "Huge potholes on the highway are causing accidents.",
    "Power supply has been interrupted since yesterday.",
    "Someone demanded a bribe to approve my building permit.",
    "Government hospital has no doctors available.",
    "Traffic signal is not functioning at the main junction.",
    "The public bus is always overcrowded and arrives late.",
    "Illegal dumping of waste is polluting the nearby lake.",
    "School building ceiling is damaged and unsafe.",
    "Stray dogs are attacking people in our locality."
]

for complaint in test_complaints:
    result = predict_department(complaint)
    print("=" * 70)
    print("Complaint :", complaint)
    print("Prediction:", result["department"])
    print("Confidence:", f'{result["confidence"]}%')

In [ ]:
long_complaint = """
For the last ten days there has been no drinking water supply in our area.
Many elderly people and children are suffering because we have to buy water
from private tankers every day. We have complained several times to the local
office but no action has been taken yet. Kindly resolve this issue immediately.
"""

result = predict_department(long_complaint)

print(result)

In [ ]:
ambiguous = [
    "There is waterlogging because the drainage is blocked.",
    "The road is damaged and there is no street light.",
    "Garbage is blocking the road.",
    "Electric pole has fallen after heavy rain.",
    "Traffic signal and street lights are not working."
]

for complaint in ambiguous:
    result = predict_department(complaint)
    print("=" * 70)
    print(complaint)
    print(result)